In [0]:
# 1. CSV
# 2. Parquet  --> par-k, par-ket  |  highly compressed file format | snappy | data analytics | predicate pushdown
# 3. Avro --> streaming data | support schema evolution | predicate pushdown
# 4. Delta tables --> DeltaLake --> Datalake --> Delta -> delta log --> ACID | Schema Evolution | time travel 

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
file_path = "/Volumes/workspace/default/kaggle_datasets/employee_salary_dataset.csv"

# emp_df = spark.read.csv(file_path, header=True, inferSchema=True)
emp_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(file_path)
display(emp_df)

In [0]:
windowSpec = Window.partitionBy("Department").orderBy(F.col("Monthly_Salary").desc())

emp_df = (
    emp_df
    .withColumn("rn", F.row_number().over(windowSpec))
    # .filter(F.col("rn") == 1)
    .select(
        "EmployeeID",
        "Name",
        "Department",
        "Monthly_Salary",
        "rn"
    )

)
emp_df.display()

In [0]:
# write the df into parquet file format

output_file_path = "/Volumes/workspace/default/kaggle_datasets/employee_dataset"
emp_df.write.format("parquet").mode("overwrite").save(output_file_path)

In [0]:
file_path = "/Volumes/workspace/default/kaggle_datasets/DATAACT_EIDL_LOANS_20200401-20200609.csv"
output_path = "/Volumes/workspace/default/kaggle_datasets/Loans"

loans_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(file_path)
# loans_df.write.format("parquet").mode("overwrite").save(output_path)

In [0]:
loans_df.rdd.getNumPartitions()

In [0]:
loans_df = loans_df.repartition(2)
loans_df.write.format("parquet").mode("overwrite").save(output_path)

In [0]:
loans_df = loans_df.coalesce(1)
loans_df.write.format("parquet").mode("overwrite").save(output_path)

In [0]:
# 1. repartition --> is used to either increase the no.of partitions or decerease the no of partition
# 2. coalesce --> is used to decrease the no of partitions 

In [0]:
loans_df = spark.read.format("parquet").load(output_path)
loans_df.display()